In [ ]:
!pip -q install -U chronos-forecasting peft wandb pyarrow
!pip -q uninstall -y torchao || true
import os, gc, numpy as np, pandas as pd, torch
print("CUDA:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/data/taxi_series.parquet"
FINAL_DIR = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/chronos_ft_final"
RESOLUTIONS = []
os.makedirs(FINAL_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
if RESOLUTIONS:
    df = df[df.resolution.isin(RESOLUTIONS)].copy()
df["ts"] = pd.to_datetime(df["ts"]); df = df.sort_values(["series_id","split","ts"])
def to_dict(sp): return {s: g.sort_values("ts")["value"].to_numpy(np.float32)
                         for s, g in df[df.split==sp].groupby("series_id")}
TRAIN, VAL = to_dict("train"), to_dict("val")
META = df[["series_id","metric","resolution","vendor"]].drop_duplicates().set_index("series_id")
FULL = {s: np.concatenate([TRAIN[s], VAL.get(s, np.array([],np.float32))]) for s in TRAIN}
VAL_START = {s: len(TRAIN[s]) for s in TRAIN}
train_inputs = [{"target": TRAIN[s]} for s in TRAIN if len(TRAIN[s]) >= 512 + 24]
print("series:", len(TRAIN), "| train_inputs:", len(train_inputs))

In [ ]:
CONTEXT, HORIZON, EVAL_MAX_WINDOWS = 512, 24, 150
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def wape(y,p):
    y,p=np.asarray(y,float),np.asarray(p,float); d=np.abs(y).sum()
    return float(np.abs(y-p).sum()/d*100) if d else float("nan")

@torch.no_grad()
def _median_forecast(pipe, ctx_batch, H):
    x = torch.tensor(np.asarray(ctx_batch, np.float32)[:, None, :], dtype=torch.float32)
    q, _ = pipe.predict_quantiles(x, prediction_length=H, quantile_levels=[0.5])
    if isinstance(q, (list, tuple)):
        q = np.stack([a.float().cpu().numpy() if hasattr(a,"cpu") else np.asarray(a,float) for a in q], 0)
    else:
        q = q.float().cpu().numpy() if hasattr(q,"cpu") else np.asarray(q,float)
    return np.asarray(q, float).reshape(x.shape[0], -1)[:, :H]

@torch.no_grad()
def eval_wape(pipe, batch=64):
    windows=[]
    for sid, arr in FULL.items():
        vs=VAL_START[sid]; origins=[o for o in range(vs,len(arr)-HORIZON+1,HORIZON) if o-CONTEXT>=0]
        if len(origins)>EVAL_MAX_WINDOWS:
            origins=[origins[i] for i in np.linspace(0,len(origins)-1,EVAL_MAX_WINDOWS).astype(int)]
        for o in origins: windows.append((sid,arr[o-CONTEXT:o],arr[o:o+HORIZON]))
    per={}
    for i in range(0,len(windows),batch):
        ch=windows[i:i+batch]; mp=_median_forecast(pipe, np.stack([c for _,c,_ in ch]), HORIZON)
        for j,(sid,_,tgt) in enumerate(ch):
            per.setdefault(sid,([],[])); per[sid][0].append(tgt); per[sid][1].append(mp[j])
    per_res={}
    for sid,(ys,ps) in per.items():
        w=wape(np.concatenate(ys),np.concatenate(ps)); r=META.loc[sid].resolution
        per_res.setdefault(r,[]).append(w)
    return per_res


In [ ]:
from chronos import BaseChronosPipeline

# REPLACE with your Optuna study.best_params
BEST_PARAMS = {"lr": 0.0003869884753980149, "lora_r": 8, "lora_alpha": 64}
# -------------------------------------------

LORA_TARGETS = ["self_attention.q","self_attention.v","self_attention.k",
                "self_attention.o","output_patch_embedding.output_layer"]
NUM_STEPS, BATCH = 4000, 32     # all-res → more data, a few thousand steps

BASE = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2", device_map=DEVICE,
    torch_dtype=torch.bfloat16 if DEVICE=="cuda" else torch.float32)

best = BASE.fit(
    train_inputs, prediction_length=HORIZON, finetune_mode="lora",
    lora_config={"r": BEST_PARAMS["lora_r"], "lora_alpha": BEST_PARAMS["lora_alpha"],
                 "target_modules": LORA_TARGETS},
    context_length=CONTEXT, learning_rate=BEST_PARAMS["lr"], num_steps=NUM_STEPS,
    batch_size=BATCH, output_dir=FINAL_DIR, finetuned_ckpt_name="checkpoint", disable_tqdm=True)

# per-resolution holdout WAPE
per_res = eval_wape(best)
print("=== FINAL all-res holdout WAPE (per resolution) ===")
for r in sorted(per_res): print(f"  {r:>4}: {np.nanmedian(per_res[r]):.3f}")
allv = [w for ws in per_res.values() for w in ws]
print("overall median WAPE:", round(float(np.nanmedian(allv)), 3))
print("saved →", os.path.join(FINAL_DIR, "checkpoint"))
